In [ ]:
import os
import json
import pickle
import sympy as sp
import pandas as pd
from IPython.display import display, HTML

In [ ]:
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']

# Variables: (term order, LaTeX label)
VARS = {
    'bl':         (0, r'\widetilde{\mathrm{B_L}}'),
    'rh':         (1, r'\widehat{\mathrm{RH}}'),
    'thetae':     (2, r'\widehat{\theta}_{e}'),
    'thetaestar': (3, r'{\widehat{\theta}_{e}^{*}}'),
    'lf':         (4, r'\widetilde{\mathrm{LF}}'),
    'shf':        (5, r'\widetilde{\mathrm{SHF}}'),
    'lhf':        (6, r'\widetilde{\mathrm{LHF}}'),
}
SYMS  = {k: sp.Symbol(k) for k in VARS}
FUNCS = {
    'min': sp.Min, 'max': sp.Max, 'abs': sp.Abs,
    'neg': lambda x: -x, 'square': lambda x: x**2,
    'cube': lambda x: x**3, 'exp': sp.exp,
}

# Greek constants: (HTML label, sympy symbol)
GREEK = {
    'alpha': ('\u03b1',                  sp.Symbol('alpha')),
    'beta':  ('\u03b2',                  sp.Symbol('beta')),
    'beta1': ('\u03b2<sub>1</sub>',      sp.Symbol('beta_1')),
    'beta2': ('\u03b2<sub>2</sub>',      sp.Symbol('beta_2')),
    'gamma': ('\u03b3',                  sp.Symbol('gamma')),
    'eta':   ('\u03b7',                  sp.Symbol('eta')),
    'delta': ('\u03b4',                  sp.Symbol('delta')),
}

# Model constants: optimizer key -> Greek name
CONST_MAP = {
    'sr_lo':  {'a': 'alpha', 'b': 'beta'},
    'sr_bl':  {'a': 'beta1', 'b': 'beta2'},
    'sr_med': {'a': 'alpha', 'b': 'gamma', 'c': 'eta'},
    'sr_hi':  {'a': 'alpha', 'b': 'gamma', 'c': 'eta', 'd': 'delta'},
}

def term_key(term):
    syms = term.free_symbols
    if not syms: return (99, str(term))
    return (min(VARS.get(s.name, (50,))[0] for s in syms), str(term))

def order_expr(expr):
    if expr.args:
        expr = expr.func(*[order_expr(a) for a in expr.args], evaluate=False)
    if isinstance(expr, sp.Add):
        expr = sp.Add(*sorted(sp.Add.make_args(expr), key=term_key), evaluate=False)
    return expr

def to_latex(expr):
    latex = sp.latex(expr, symbol_names={SYMS[k]: v[1] for k, v in VARS.items()}, mul_symbol='dot')
    return ' '.join(latex.replace(r'\left', '').replace(r'\right', '').split())

_regpath = os.path.join(MODELSDIR, 'sr', 'optimized_equations.pkl')
registry = pickle.load(open(_regpath, 'rb')) if os.path.exists(_regpath) else {}

In [ ]:
rows1, rows2 = [], []
for name in ['sr_lo', 'sr_bl', 'sr_med', 'sr_hi']:
    eqspec = SRCONFIG['optimizedeqs'].get(name)
    if eqspec is None: continue
    opt  = registry.get(name)
    desc = eqspec['description']

    parts = [f"{GREEK[g][0]} = {opt['constants'][k]:.4f}"
             for k, g in CONST_MAP[name].items() if opt and k in opt['constants']]
    rows1.append({'Equation': desc, 'Constants': ', '.join(parts) or '\u2013'})

    if opt:
        ns = {**SYMS, **FUNCS, **{k: GREEK[g][1] for k, g in CONST_MAP[name].items()}}
        try:    form = '$' + to_latex(order_expr(eval(opt['form'], {'__builtins__': {}}, ns))) + '$'
        except Exception as e: form = f"{opt['form']}  [error: {e}]"
    else:
        form = '\u2013'
    rows2.append({'Equation': desc, 'Form': form})

display(HTML(pd.DataFrame(rows1).set_index('Equation').to_html(escape=False)))
_css = '<style>.eq-tbl td { text-align: left; }</style>'
display(HTML(_css + pd.DataFrame(rows2).set_index('Equation').to_html(escape=False, classes='eq-tbl')))